Recommendation System using TF-IDF and Cosine Similarity

1. Create a dataset 
2. Preprocess text (lowercase, punctuation, stopwords, lemmatization)
3. Convert text to TF-IDF vectors
4. Accept user query and preprocess it
5. Calculate cosine similarity
6. Return top 3 most similar results

In [1]:
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import string

 Step 1: Create Dataset

In [2]:

movies_dataset = [
    {
        'title': 'Interstellar',
        'description': 'A team of astronauts travels through a wormhole in space to find a new habitable planet for humanity. They encounter black holes, time dilation, and robotic companions on their journey across the galaxy.'
    },
    {
        'title': 'Star Wars: A New Hope',
        'description': 'Luke Skywalker joins forces with a smuggler and his android companions to rescue a princess and defeat the evil galactic empire using laser weapons and space ships.'
    },
    {
        'title': 'The Martian',
        'description': 'An astronaut stranded on Mars must use his engineering skills and ingenuity to survive on the alien planet while waiting for rescue from Earth. A story of human resilience and problem-solving.'
    },
    {
        'title': 'Blade Runner',
        'description': 'A detective hunts down artificial humanoids called replicants in a dystopian future city. A noir thriller about what it means to be human in a world with advanced robotics and AI.'
    },
    {
        'title': 'Avatar',
        'description': 'A paraplegic marine travels to an alien moon and infiltrates a blue-skinned indigenous civilization to gather intelligence. He falls in love with the alien culture and fights against corporate exploitation.'
    },
    {
        'title': 'The Iron Giant',
        'description': 'A young boy befriends a massive alien robot that falls to Earth. Together they explore friendship, acceptance, and what it means to choose peace over war in a cold war era setting.'
    },
    {
        'title': 'Ex Machina',
        'description': 'A programmer is invited to test an advanced artificial intelligence robot with human-like appearance. A psychological thriller about consciousness, manipulation, and the nature of intelligence.'
    },
    {
        'title': 'Gravity',
        'description': 'An astronaut is stranded in space after debris destroys her spacecraft. Floating through the void, she must navigate the dangers of the cosmos and find a way back to Earth.'
    },
    {
        'title': 'Transformers',
        'description': 'Giant alien robots known as Transformers battle on Earth, hiding in disguise as vehicles. Humans team up with heroic robots to fight against evil mechanical invaders from outer space.'
    },
    {
        'title': 'The Fifth Element',
        'description': 'In a futuristic space metropolis, a taxi driver and a mysterious alien woman must retrieve four ancient elements to save Earth from destruction by an overwhelming cosmic force.'
    }
]


 Step 2: Create Text Preprocessing

In [3]:
lemmatizer = WordNetLemmatizer()
punkt=string.punctuation
stop_words = set(stopwords.words('english'))
def preprocess(text):
        
        text = text.lower()
 
        text =''.join(char for char in text if char not in punkt)
        
        tokens = word_tokenize(text)
        
        tokens = [
            lemmatizer.lemmatize(token)
            for token in tokens
            if token not in stop_words
        ]
        
        return ' '.join(tokens)

sample_text = "A team of astronauts travels through a wormhole in space!"
print(f"Original: {sample_text}")
print(f"Processed: {preprocess(sample_text)}")

Original: A team of astronauts travels through a wormhole in space!
Processed: team astronaut travel wormhole space



 Step 3: Preprocess All Documents

In [4]:
descriptions = [doc['description'] for doc in movies_dataset]
processed_descriptions = [preprocess(desc) for desc in descriptions]

 Step 4: Convert Text to TF-IDF Vectors

In [ ]:
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(processed_descriptions)

print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
print(f"Number of documents: {tfidf_matrix.shape[0]}")

TF-IDF Matrix Shape: (10, 141)
Number of documents: 10


 Step 5: Create Search Function with Cosine Similarity

In [ ]:
def search(query, top_k=3):
    processed_query = preprocess(query)
    query_vector = vectorizer.transform([processed_query])
    scores = cosine_similarity(query_vector, tfidf_matrix)[0]
    top_indices = np.argsort(scores)[::-1][:top_k]

    results = [
        {
            "title":            movies_dataset[i]["title"],
            "description":      movies_dataset[i]["description"],
            "similarity_score": round(float(scores[i]), 4),
        }
        for i in top_indices
    ]
    return results


def print_results(query, results):
    print(f"\nQuery: \"{query}\"")
    print("-" * 65)
    for rank, r in enumerate(results, 1):
        print(f"{rank}. {r['title']}")
        print(f"   Score      : {r['similarity_score']:.4f}")
        print(f"   Description: {r['description'][:90]}...")
        print('\n')

 Step 6: 3 results

In [35]:
query = "Artificial Intelligence"
results = search(query)
print_results(query, results)


Query: "Artificial Intelligence"
-----------------------------------------------------------------
1. Ex Machina
   Score      : 0.4718
   Description: A programmer is invited to test an advanced artificial intelligence robot with human-like ...


2. Blade Runner
   Score      : 0.1530
   Description: A detective hunts down artificial humanoids called replicants in a dystopian future city. ...


3. Avatar
   Score      : 0.1489
   Description: A paraplegic marine travels to an alien moon and infiltrates a blue-skinned indigenous civ...




In [38]:
query = "Happy new year"
results = search(query, top_k=3)
print_results(query, results)


Query: "Happy new year"
-----------------------------------------------------------------
1. Interstellar
   Score      : 0.2389
   Description: A team of astronauts travels through a wormhole in space to find a new habitable planet fo...


2. The Fifth Element
   Score      : 0.0000
   Description: In a futuristic space metropolis, a taxi driver and a mysterious alien woman must retrieve...


3. Transformers
   Score      : 0.0000
   Description: Giant alien robots known as Transformers battle on Earth, hiding in disguise as vehicles. ...




In [39]:
query = "is the earth flat"
results = search(query, top_k=3)
print_results(query, results)


Query: "is the earth flat"
-----------------------------------------------------------------
1. Gravity
   Score      : 0.1631
   Description: An astronaut is stranded in space after debris destroys her spacecraft. Floating through t...


2. The Martian
   Score      : 0.1558
   Description: An astronaut stranded on Mars must use his engineering skills and ingenuity to survive on ...


3. The Fifth Element
   Score      : 0.1469
   Description: In a futuristic space metropolis, a taxi driver and a mysterious alien woman must retrieve...


